In [1]:
# Mount Google Drive to save everything permanently
from google.colab import drive
drive.mount('/content/drive')

# Create your project folder
!mkdir -p "/content/drive/MyDrive/CFT_Project/notebooks"
!mkdir -p "/content/drive/MyDrive/CFT_Project/results"

print("✅ Google Drive mounted!")
print("Your work will auto-save to: /content/drive/MyDrive/CFT_Project/")

Mounted at /content/drive
✅ Google Drive mounted!
Your work will auto-save to: /content/drive/MyDrive/CFT_Project/


In [2]:
# CFT v3.1 — Scaled for Colab GPU (Real-World Rule Task)
import random, torch, torch.nn as nn, torch.nn.functional as F

torch.manual_seed(42)
random.seed(42)

# === REAL-WORLD VOCABULARY (30 animals — too many to memorize!) ===
ANIMALS = ["cat", "dog", "bird", "fish", "lion", "tiger", "bear", "wolf", "fox", "deer",
           "rabbit", "mouse", "horse", "cow", "pig", "sheep", "goat", "monkey", "ape", "whale",
           "shark", "eagle", "hawk", "owl", "penguin", "frog", "snake", "turtle", "lizard", "ant"]
VOCAB = ANIMALS + ["is_a", "?", "yes", "no", "<pad>"]
TOK = {word: i+1 for i, word in enumerate(VOCAB)}

# === BIGGER MODEL (Colab GPU can handle this) ===
D, N_SLOTS = 128, 16  # 2x bigger than laptop version

# === DATA GENERATOR (Random chains — can't memorize!) ===
def make_rule_sample():
    chain = random.sample(ANIMALS, 3)  # e.g., ["lion", "penguin", "ant"]
    facts = [(chain[0], chain[1]), (chain[1], chain[2])]
    is_true = random.random() < 0.5
    qa, qb = (chain[0], chain[2]) if is_true else (chain[2], chain[0])
    label = 1 if is_true else 0
    return facts, (qa, qb), label

def batch(bs, num_facts):
    samples = [make_rule_sample() for _ in range(bs)]
    chunks, types = [], []
    for k in range(num_facts):
        chunk_data = [[TOK[f[k][0]], TOK["is_a"], TOK[f[k][1]]] for f, _, _ in samples]
        chunks.append(torch.tensor(chunk_data))
        types.append(torch.full((bs, 3), 1).long())
    query_data = [[TOK[q[0]], TOK["?"], TOK[q[1]]] for _, q, _ in samples]
    chunks.append(torch.tensor(query_data))
    types.append(torch.full((bs, 3), 2).long())
    labels = torch.tensor([l for *_, l in samples])
    return chunks, types, labels

# === THE MODEL (Same architecture, bigger) ===
class BDHCQ(nn.Module):
    def __init__(self):
        super().__init__()
        self.S = nn.Parameter(torch.randn(N_SLOTS, D)*.02)
        self.wk, self.wv, self.wr, self.wl = (nn.Linear(D, D, bias=False) for _ in range(4))
        self.wg, self.wg2, self.wo = nn.Linear(D, D), nn.Linear(D, D), nn.Linear(D, D, bias=False)
    def parts(self, x):
        A = torch.softmax(self.S @ self.wk(x).transpose(1, 2)/D**.5, -1)
        P = A @ self.wv(x); R = self.wl(P)
        return P + torch.softmax(self.wr(P) @ R.transpose(1, 2)/D**.5, -1) @ R
    def write(self, Pr, M, g):
        Md = M * g
        Aw = torch.softmax(Pr @ Md.transpose(1, 2)/D**.5, -1)
        Upd = Aw.transpose(1, 2) @ self.wo(Pr)
        return Md + torch.sigmoid(self.wg(Upd) + self.wg2(Md)) * Upd

class CFTRuleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(len(VOCAB)+1, D)
        self.typ = nn.Embedding(3, D)
        self.pos = nn.Embedding(3, D)
        self.local = nn.TransformerEncoderLayer(D, 4, 128, dropout=0.0, batch_first=True)
        self.bdh = BDHCQ()
        self.M0 = nn.Parameter(torch.randn(N_SLOTS, D)*.1)
        self.g = nn.Parameter(torch.zeros(N_SLOTS, 1))
        self.rq = nn.Linear(D, D, bias=False)
        self.head = nn.Sequential(nn.Linear(2*D, 64), nn.ReLU(), nn.Linear(64, 2))
        self.fid = nn.Linear(N_SLOTS*D, D)
    def embed(self, c, t):
        return self.tok(c) + self.typ(t) + self.pos(torch.arange(c.size(1), device=c.device))
    def forward(self, chunks, types, ablate=False):
        B = chunks[0].size(0)
        M = self.M0.unsqueeze(0).expand(B, -1, -1)
        g = torch.sigmoid(self.g)
        pools = []
        for c, t in zip(chunks[:-1], types[:-1]):
            x = self.local(self.embed(c, t)); pools.append(x.mean(1))
            M = self.bdh.write(self.bdh.parts(x), M, g)
        Pq = self.bdh.parts(self.local(self.embed(chunks[-1], types[-1])))
        Mr = torch.zeros_like(M) if ablate else M
        A = torch.softmax(self.rq(Pq) @ Mr.transpose(1, 2)/D**.5, -1)
        logits = self.head(torch.cat([Pq.mean(1), (A @ Mr).mean(1)], -1))
        return logits, self.fid(M.flatten(1)), torch.stack(pools, 1).mean(1).detach()

# === TRAINING (Developmental Curriculum) ===
model = CFTRuleModel().cuda()  # Move to GPU!
print(f"Model Parameters: {sum(p.numel() for p in model.parameters()):,}")

opt = torch.optim.Adam(model.parameters(), 1e-3)
PHASES = [(1, 1500), (2, 2000)]  # Phase 1: 1 fact, Phase 2: 2 facts

for phase, (num_facts, steps) in enumerate(PHASES, 1):
    print(f"\n--- Starting Phase {phase} ({num_facts} facts, {steps} steps) ---")
    for step in range(steps):
        chunks, types, y = batch(128, num_facts)
        chunks = [c.cuda() for c in chunks]  # Move data to GPU
        types = [t.cuda() for t in types]
        y = y.cuda()
        logits, recon, tgt = model(chunks, types)
        loss = F.cross_entropy(logits, y) + 0.3 * F.mse_loss(recon, tgt)
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 500 == 0:
            print(f"  step {step:4d}/{steps} | Loss: {loss.item():.3f}")

# === TEST + ABLATION PROOF ===
@torch.no_grad()
def test_acc(num_facts, ablate=False, n=400):
    chunks, types, y = batch(n, num_facts)
    chunks = [c.cuda() for c in chunks]
    types = [t.cuda() for t in types]
    y = y.cuda()
    lo, _, _ = model(chunks, types, ablate=ablate)
    return (lo.argmax(1) == y).float().mean().item()

print("\n" + "="*50)
print("FINAL REAL-WORLD RESULTS")
print("="*50)
print(f"1-Fact Query Accuracy:   {test_acc(1)*100:.1f}%")
print(f"2-Fact Query Accuracy:   {test_acc(2)*100:.1f}%  <-- (Lion→Penguin→Ant)")
print(f"2-Fact ABLATED Accuracy: {test_acc(2, ablate=True)*100:.1f}%  <-- (Memory erased)")
print("\nIf ablated drops to ~50%, it PROVES reasoning happened in Fleeting Memory!")

Model Parameters: 519,250

--- Starting Phase 1 (1 facts, 1500 steps) ---
  step    0/1500 | Loss: 0.885
  step  500/1500 | Loss: 0.044
  step 1000/1500 | Loss: 0.067

--- Starting Phase 2 (2 facts, 2000 steps) ---
  step    0/2000 | Loss: 5.424
  step  500/2000 | Loss: 0.088
  step 1000/2000 | Loss: 0.050
  step 1500/2000 | Loss: 0.029

FINAL REAL-WORLD RESULTS
1-Fact Query Accuracy:   90.5%
2-Fact Query Accuracy:   97.5%  <-- (Lion→Penguin→Ant)
2-Fact ABLATED Accuracy: 51.0%  <-- (Memory erased)

If ablated drops to ~50%, it PROVES reasoning happened in Fleeting Memory!
